# KRI/KPI Context Builder
Configure your runs below, then execute the pipeline cell.

In [ ]:
# ── INPUT PARAMETERS ─────────────────────────────────────────────

INGESTION_QUARTER = "Q1_2026"

# List of (country, business_line) combinations to process
RUNS = [
    ("PL", "RB"),
    ("RO", "WB"),
    # ("FR", "RB"),
    # ("CH", "RB"),
]

INPUT_DIR  = "input/"
OUTPUT_DIR = "output/"

In [ ]:
# ── RUN PIPELINE ─────────────────────────────────────────────────

from core import resolve_quarter, load_tables, filter_kris, enrich_kpis, build_output

qi = resolve_quarter(INGESTION_QUARTER)
print(f"Quarters: ingestion={qi.ingestion}  test={qi.test}  base={qi.base}\n")

for country, bl in RUNS:
    print(f"{'='*60}")
    print(f"  Processing {country}/{bl}")
    print(f"{'='*60}")
    try:
        tables = load_tables(INPUT_DIR, country, bl)
        kri_results = filter_kris(tables, qi)
        ads = set(kri_results)
        print(f"  → {len(ads)} alert definition(s) with triggered KRI(s)")
        kpi_data, kpi_avail = enrich_kpis(tables, ads, qi)
        build_output(kri_results, kpi_data, kpi_avail, qi, OUTPUT_DIR, country, bl)
    except Exception as e:
        print(f"  ✗ Failed: {e}")
    print()